# Notebook 09 — Validation suite across all 5 model/task settings

Runs the **same quantitative validation / baseline / ablation suite** as notebook 08, but parametrically across every setting in the paper:

| `NB09_EXP` | model | task |
|---|---|---|
| `mlp_even_odd` | SimpleMLP 784→8→4→2 | MNIST {0,1,3,4} even/odd |
| `mlp_digit` | SimpleMLP 784→40→20→10 | MNIST 10-digit |
| `cnn_cifar` | SmallCNN | CIFAR-10 |
| `vit_mnist` | TinyViT | MNIST even/odd |
| `imagenet_cnn` | SqueezeNet 1.1 (pretrained) | ImageNet (8 super-categories) |

One execution = one experiment. Select with the `NB09_EXP` env var and the compute profile with `NB09_MODE` (`local` | `cluster`).

The **qualitative circuit figures** (scaffolds, pixel RFs, factor panels) live in the per-architecture notebooks 01–05; this notebook is the uniform *quantitative* validation.

Sections are **capability-gated** — anything an architecture cannot support is skipped with a recorded reason rather than failing (e.g. causal reconstruction needs primary-mode BFT, so layer-dict architectures skip it).

⚠️ **See the final cell for the validation-status table** — only the two MLP settings have actually been executed; CNN/ViT/ImageNet are written from the notebook 03/04/05 specs and are untested.

## §0 · Setup, compute profile & helpers

In [1]:
import os, sys, json, copy, warnings
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from scipy.stats import wilcoxon, ttest_rel
from sklearn.decomposition import MiniBatchNMF
from sklearn.manifold import MDS
from sklearn.metrics import silhouette_score, roc_auc_score
from sklearn.metrics.pairwise import cosine_distances, paired_cosine_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

from src import (SimpleMLP, load_experiment, save_experiment, get_transform,
                 get_loaders_from_config, collect_layer_dicts, bft, evaluate,
                 select_class_circuit, per_class_accuracy,
                 extract_fingerprint_matrix, project_stimuli_onto_tree,
                 compute_nmf_stability, compute_k_sensitivity)
from src.bft import (compute_joint_arbors_normalized, compute_conv_joint_arbors,
                     compute_attn_joint_arbors)
from src.training import train_epoch, label_transform_even_odd
from src.data_utils import get_mnist_loaders, label_transformed_loader
from src.robustness_utils import align_factors

warnings.filterwarnings('ignore')
RNG  = np.random.default_rng(0)
REPO = os.path.abspath('..')                     # notebook lives in notebooks/
EXP  = os.environ.get('NB09_EXP',  'mlp_even_odd')
MODE = os.environ.get('NB09_MODE', 'local')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

FIG_DIR    = os.path.join(REPO, 'figs', '09_validation', EXP)
RES_DIR    = os.path.join(REPO, 'data', 'results')
MODEL_ROOT = os.path.join(REPO, 'data', 'models')
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

# Compute profile. 'local' caps everything for laptop iteration; 'cluster' is the
# full run (use a GPU node — see the final cell for job scripts).
if MODE == 'cluster':
    N_TRACE, BFT_MAX_ITER, AUX_MAX_ITER, STAB_SEEDS, NMF_SUB, IG_STEPS = None, 500, 300, 10, 800, 48
else:
    N_TRACE, BFT_MAX_ITER, AUX_MAX_ITER, STAB_SEEDS, NMF_SUB, IG_STEPS = 400, 100, 80, 4, 250, 8

results, figpaths = {'experiment': EXP, 'mode': MODE}, {}


def savefig(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, bbox_inches='tight')
    figpaths[name] = os.path.relpath(p, REPO)
    print('  saved', figpaths[name])
    plt.show()


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    return o if isinstance(o, (float, int, str, bool)) or o is None else str(o)


def sub_rows(X):
    return X if X.shape[0] <= NMF_SUB else X[RNG.choice(X.shape[0], NMF_SUB, replace=False)]


def stab_mean(X, k):
    sim, _ = compute_nmf_stability(sub_rows(X), k, n_seeds=STAB_SEEDS, max_iter=AUX_MAX_ITER)
    return sim[~np.eye(sim.shape[0], dtype=bool)]


def knn_cv(X, y, kmax=5, cv_max=3):
    """kNN cross-val accuracy, robust to non-contiguous and small classes.

    Counts only classes that are PRESENT: np.bincount() reports a 0 for gaps in the
    label set (e.g. digits {0,1,3,4} -> a zero at index 2), which would yield
    n_neighbors=0 and make sklearn raise.
    """
    classes, counts = np.unique(y, return_counts=True)
    mn = int(counts.min())
    if len(classes) < 2 or mn < 2:
        return float('nan')
    cv = int(min(cv_max, mn))              # StratifiedKFold needs >= cv per class
    k = int(max(1, min(kmax, mn)))         # and n_neighbors >= 1
    return float(cross_val_score(KNeighborsClassifier(k), X, y, cv=cv).mean())


def sep_metrics(X, y):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    if len(np.unique(y)) < 2 or len(y) < 12:
        return float('nan'), float('nan')
    return float(silhouette_score(Xn, y)), knn_cv(Xn, y)


def fit_nmf(X, k):
    """Fit on a row subsample (speed), transform the full matrix for loadings."""
    Xc = np.clip(X, 0, None).astype(np.float32)
    m = MiniBatchNMF(n_components=k, random_state=0, max_iter=AUX_MAX_ITER,
                     batch_size=1024, init='random')
    m.fit(sub_rows(Xc))
    return m.transform(Xc), m.components_.T


print(f'EXP={EXP}  MODE={MODE}  DEVICE={DEVICE}  N_TRACE={N_TRACE}')

/home/jb3879/Factor_Trace/.venv2/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


EXP=vit_mnist  MODE=cluster  DEVICE=cpu  N_TRACE=None


## §1 · Experiment registry

Per-experiment model/data/BFT hyperparameters, taken from notebooks 01–05.

In [2]:
REG = {
    'mlp_even_odd': dict(kind='mlp', ckpt='mnist_even_odd_mlp_8_4_0134', n_seeds=5,
        arch_kwargs=dict(input_dim=784, hidden_dims=[8, 4], output_dim=2),
        digit_filter=[0, 1, 3, 4], label='even_odd', n_classes=2,
        class_names={0: 'even', 1: 'odd'},
        bft=dict(k_max=[5, 5, 5], n_branches=[1, 1, 2], stimulus_threshold=0.5)),
    'mlp_digit': dict(kind='mlp', ckpt='mnist_digit_mlp_40_20', n_seeds=5,
        arch_kwargs=dict(input_dim=784, hidden_dims=[40, 20], output_dim=10),
        digit_filter=None, label='identity', n_classes=10,
        class_names={i: str(i) for i in range(10)},
        bft=dict(k_max=[10, 6, 10], n_branches=[1, 1, 10], stimulus_threshold=0.7)),
    'cnn_cifar': dict(kind='cnn', ckpt='cifar10_cnn', n_seeds=5, conf_per_class=60,
        arch_kwargs=dict(channels=[32, 64, 128, 256], n_classes=10, global_pool=True),
        label='identity', n_classes=10,
        class_names={i: n for i, n in enumerate(
            ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'])},
        bft=dict(k_max=[6, 6, 6, 6, 10], n_branches=[1, 1, 1, 1, 10],
                 conv_pool_method='avg', stimulus_threshold=0.0)),
    'vit_mnist': dict(kind='vit', ckpt='mnist_even_odd_vit_tiny', n_seeds=5,
        arch_kwargs=dict(embed_dim=32, n_heads=2, ffn_dim=64, n_classes=2),
        label='even_odd', n_classes=2, class_names={0: 'even', 1: 'odd'},
        bft=dict(k_max=[10, 6, 6, 4], n_branches=[1, 1, 2, 4], stimulus_threshold=0.0)),
    'imagenet_cnn': dict(kind='imagenet', conf_per_class=50, n_classes=8,
        class_names={i: n for i, n in enumerate(
            ['airplane', 'ship', 'car', 'bicycle', 'elephant', 'bear', 'dog', 'bird'])},
        bft=dict(k_max=[4] * 9 + [8], n_branches=[1] * 8 + [2, 5],
                 conv_pool_method='avg', stimulus_threshold=0.0)),
}


def confidence_filter(raw, per_class):
    """Keep the top-`per_class` most confident correct samples per class."""
    tgt, conf, keep = raw['targets'], raw['confidences'], []
    for c in np.unique(tgt):
        idx = np.where(tgt == c)[0]
        keep.extend(idx[np.argsort(conf[idx])[::-1][:per_class]])
    keep = np.array(sorted(keep))
    out = dict(raw)
    for k in ('images', 'targets', 'confidences', 'digits'):
        if k in raw:
            out[k] = raw[k][keep]
    out['layer_data'] = [dict(d, input_fmap=d['input_fmap'][keep]) for d in raw['layer_data']]
    return out

## §2 · Build the experiment (model → BFT tree)

Handles primary-mode vs layer-dict BFT, confidence pre-filtering, CLS-token extraction (ViT) and spine-layer filtering (SqueezeNet), and sets the capability flags.

In [3]:
def build_experiment(exp):
    """Return a ctx dict: model, BFT tree, samples, layer inputs, and capability flags.

    caps gate the sections that are not architecture-general:
      recon      – needs primary-mode BFT (model hook); layer-dict archs cannot
      roundtrip  – NNLS projection; not supported through 'attn' nodes
      ablation   – needs BFT node layer_name to map onto a model weight module
      bft_pixel  – input-layer factors reshapeable to pixels (MLP only)
      multi_seed – >=2 seed checkpoints on disk
    """
    r = REG[exp]
    ctx = dict(exp=exp, n_classes=r['n_classes'], class_names=r['class_names'],
               caps=dict(recon=False, roundtrip=False, ablation=False,
                         attribution=False, bft_pixel=False, multi_seed=False))

    if r['kind'] == 'mlp':
        cfg = {'arch': 'SimpleMLP', 'arch_kwargs': r['arch_kwargs'], 'dataset': 'MNIST',
               'dataset_kwargs': {'root': '../data/', 'batch_size': 64,
                                  **({'digit_filter': r['digit_filter']} if r['digit_filter'] else {})},
               'label_transform': r['label'], 'analysis_layer_indices': [2, 4, 6], 'n_per_class': 1000}
        ltf = get_transform(r['label'])
        seeds = {}
        for s in range(r['n_seeds']):
            ed = os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed{s}")
            if os.path.exists(os.path.join(ed, 'weights.pt')):
                seeds[s] = load_experiment(ed, DEVICE)[0]
        if 0 not in seeds:
            raise RuntimeError(f'{exp}: no seed-0 checkpoint under {MODEL_ROOT}; train it first.')
        model = seeds[0]
        _, test_loader = get_loaders_from_config(cfg)
        if N_TRACE is not None:
            test_loader = DataLoader(Subset(test_loader.dataset,
                                            list(range(min(N_TRACE, len(test_loader.dataset))))),
                                     batch_size=256, shuffle=False)
        vloader = label_transformed_loader(test_loader, ltf) if ltf else test_loader
        coll = collect_layer_dicts(model, test_loader, label_transform=ltf, device=DEVICE)
        tree = bft(model, vloader, **r['bft'], weighting='img_selectivity',
                   validate=True, max_iter=BFT_MAX_ITER, n_jobs=3)
        ctx.update(model=model, tree=tree, images=coll['images'], targets=coll['targets'],
                   fine=coll.get('digits', coll['targets']), label_transform=ltf,
                   layer_inputs=[d['input_fmap'] for d in coll['layer_data']],
                   eval_loader=test_loader, vloader=vloader, seeds=seeds, bft_kwargs=r['bft'])
        ctx['caps'].update(recon=True, roundtrip=True, ablation=True, attribution=True,
                           bft_pixel=True, multi_seed=len(seeds) >= 2)

    elif r['kind'] == 'cnn':
        from src import SmallCNN
        import torchvision.transforms as T
        from torchvision import datasets
        ed = os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed0")
        if not os.path.exists(os.path.join(ed, 'weights.pt')):
            raise RuntimeError(f'{exp}: no checkpoint at {ed}; train in notebook 03 first.')
        model = load_experiment(ed, DEVICE)[0]
        tf = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465),
                                                 (0.2470, 0.2435, 0.2616))])
        test_ds = datasets.CIFAR10('../data', train=False, download=True, transform=tf)
        test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
        raw = collect_layer_dicts(model, test_loader, device=DEVICE, only_correct=True)
        raw = confidence_filter(raw, r['conf_per_class'])
        if N_TRACE is not None and len(raw['targets']) > N_TRACE:
            keep = np.sort(RNG.choice(len(raw['targets']), N_TRACE, replace=False))
            for k in ('images', 'targets', 'confidences'):
                raw[k] = raw[k][keep]
            raw['layer_data'] = [dict(d, input_fmap=d['input_fmap'][keep]) for d in raw['layer_data']]
        tree = bft(raw['layer_data'], **r['bft'], weighting='img_selectivity',
                   max_iter=BFT_MAX_ITER, n_jobs=3)
        ctx.update(model=model, tree=tree, images=raw['images'], targets=raw['targets'],
                   fine=raw['targets'], label_transform=None,
                   layer_inputs=[d['input_fmap'] for d in raw['layer_data']],
                   eval_loader=test_loader, seeds={0: model}, bft_kwargs=r['bft'])
        ctx['caps'].update(recon=False, roundtrip=True, ablation=True, attribution=True,
                           bft_pixel=False, multi_seed=False)

    elif r['kind'] == 'vit':
        # Mirrors notebook 04: layer-dict over the CLS token; attention V-projection
        # is an 'attn' node. NOT yet validated — run on the cluster first.
        from src import TinyViT
        ed = os.path.join(MODEL_ROOT, f"{r['ckpt']}_seed0")
        if os.path.exists(os.path.join(ed, 'weights.pt')):
            model = load_experiment(ed, DEVICE)[0]
        else:
            print(f'  {exp}: no checkpoint — training seed 0 (30 epochs)')
            torch.manual_seed(0)
            model = TinyViT(**r['arch_kwargs']).to(DEVICE)
            tr, _ = get_mnist_loaders(batch_size=64, root='../data/')
            opt = torch.optim.Adam(model.parameters(), lr=1e-3)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
            for _ in range(30):
                train_epoch(model, tr, opt, nn.NLLLoss(), DEVICE, label_transform_even_odd)
                sch.step()
            save_experiment(model, {'arch': 'TinyViT', 'arch_kwargs': r['arch_kwargs'],
                                    'dataset': 'MNIST',
                                    'dataset_kwargs': {'root': '../data/', 'batch_size': 64},
                                    'label_transform': 'even_odd'}, ed)
        _, test_loader = get_mnist_loaders(batch_size=64, root='../data/')
        if N_TRACE is not None:
            test_loader = DataLoader(Subset(test_loader.dataset,
                                            list(range(min(2 * N_TRACE, len(test_loader.dataset))))),
                                     batch_size=256, shuffle=False)
        model.eval()
        IM, TG, AI, AW, AO, F1, F2 = [], [], [], [], [], [], []
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(DEVICE); yt = label_transform_even_odd(y).to(DEVICE)
                out = model(x, capture=True)
                logits = out[0] if isinstance(out, tuple) else out
                m = (logits.argmax(1) == yt)
                if not m.any():
                    continue
                blk = model.block
                IM.append(x[m].cpu().numpy()); TG.append(yt[m].cpu().numpy())
                AI.append(blk._attn_in[m].cpu().numpy())
                AW.append(blk._attn_w.mean(1)[:, 0, :][m].cpu().numpy())
                AO.append(blk._attn_out[m][:, 0].cpu().numpy())
                F1.append(blk._ffn1_in[m][:, 0].cpu().numpy())
                F2.append(blk._ffn2_in[m][:, 0].cpu().numpy())
        cat = lambda L: np.concatenate(L, 0)
        D, blk = r['arch_kwargs']['embed_dim'], model.block
        layer_dicts = [
            {'type': 'attn', 'name': 'B0-V',
             'weight': blk.attn.in_proj_weight[2 * D:, :].detach().cpu().numpy(),
             'input_fmap': cat(AI), 'attn_weights': cat(AW)},
            {'type': 'fc', 'name': 'B0-O',
             'weight': blk.attn.out_proj.weight.detach().cpu().numpy(), 'input_fmap': cat(AO)},
            {'type': 'fc', 'name': 'B0-FFN1',
             'weight': blk.ffn1.weight.detach().cpu().numpy(), 'input_fmap': cat(F1)},
            {'type': 'fc', 'name': 'B0-FFN2',
             'weight': blk.ffn2.weight.detach().cpu().numpy(), 'input_fmap': cat(F2)}]
        tree = bft(layer_dicts, **r['bft'], weighting='img_selectivity',
                   max_iter=BFT_MAX_ITER, n_jobs=3)
        tg = cat(TG)
        ctx.update(model=model, tree=tree, images=cat(IM), targets=tg, fine=tg,
                   label_transform=label_transform_even_odd,
                   layer_inputs=[d['input_fmap'] for d in layer_dicts],
                   eval_loader=test_loader, seeds={0: model}, bft_kwargs=r['bft'])
        # attn node + non-module layer names -> no recon / round-trip / name-mapped ablation
        ctx['caps'].update(recon=False, roundtrip=False, ablation=False,
                           attribution=True, bft_pixel=False, multi_seed=False)

    elif r['kind'] == 'imagenet':
        # Mirrors notebook 05: pretrained SqueezeNet 1.1, spine layers only, 8 super-
        # categories. Needs ImageNet val data + GPU. NOT yet validated — cluster only.
        import torchvision.transforms as T
        from torchvision import datasets
        from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
        CATS = {'airplane': [404, 895], 'ship': [403, 724], 'car': [609, 751],
                'bicycle': [444, 671], 'elephant': [101, 385], 'bear': [294, 297],
                'dog': [151, 251], 'bird': [7, 9]}
        idx2cat = {ii: ci for ci, c in enumerate(CATS) for ii in CATS[c]}
        model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE).eval()
        tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                        T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
        try:
            ds = datasets.ImageNet('../data', split='val', transform=tf)
            tgts = np.array(ds.targets)
        except Exception:
            root = '../data/val'
            if not os.path.isdir(root):
                raise RuntimeError('imagenet_cnn needs ImageNet val data at ../data/val '
                                   '(ImageFolder) or a torchvision ImageNet root at ../data. '
                                   'See the cluster-instructions cell.')
            ds = datasets.ImageFolder(root, transform=tf)
            tgts = np.array([t for _, t in ds.samples])
        focus = np.where(np.isin(tgts, list(idx2cat)))[0]
        floader = DataLoader(Subset(ds, focus), batch_size=64, shuffle=False, num_workers=4)

        def spine(name, mod):
            return name in ('features.0', 'classifier.1') or (
                isinstance(mod, nn.Conv2d) and name.endswith('.squeeze'))

        raw = collect_layer_dicts(model, floader, device=DEVICE, only_correct=True,
                                  layer_filter=spine)
        raw['targets'] = np.array([idx2cat[int(t)] for t in raw['targets']])   # -> 0..7
        raw = confidence_filter(raw, r['conf_per_class'])
        tree = bft(raw['layer_data'], **r['bft'], weighting='img_selectivity',
                   max_iter=BFT_MAX_ITER, n_jobs=3)
        ctx.update(model=model, tree=tree, images=raw['images'], targets=raw['targets'],
                   fine=raw['targets'],
                   label_transform=(lambda t: torch.as_tensor(
                       [idx2cat.get(int(x), -1) for x in t])),
                   layer_inputs=[d['input_fmap'] for d in raw['layer_data']],
                   eval_loader=floader, seeds={0: model}, bft_kwargs=r['bft'])
        ctx['caps'].update(recon=False, roundtrip=True, ablation=True, attribution=True,
                           bft_pixel=False, multi_seed=False)
    else:
        raise NotImplementedError(exp)
    return ctx


ctx     = build_experiment(EXP)
tree    = ctx['tree']
model   = ctx['model']
targets = ctx['targets'].astype(int)
fine    = ctx['fine'].astype(int)
layer_inputs = ctx['layer_inputs']
n_samples    = len(targets)
caps         = ctx['caps']
print(f'  built: n_samples={n_samples}  n_classes={ctx["n_classes"]}\n  caps={caps}')

nodes_by_layer = {}
for nd in tree.nodes():
    nodes_by_layer.setdefault(nd.layer_idx, nd)
layer_ids = sorted(nodes_by_layer)


def node_arbor_pos(nd):
    """Rebuild a node's positive joint arbor, dispatching on layer type.

    'attn' nodes MUST use compute_attn_joint_arbors: their input_fmap is (N, T, d_model)
    and the token axis has to be collapsed by the CLS attention weights first. Feeding
    it to the FC path broadcasts to (N, T*d, d) and then fails on the stimulus weights.
    """
    li = layer_inputs[nd.layer_idx]
    if nd.layer_type == 'conv':
        X = compute_conv_joint_arbors(nd.weight, li, stimulus_weights=nd.stimulus_weights,
                                      stimulus_threshold=nd.stimulus_threshold,
                                      pool_method=ctx['bft_kwargs'].get('conv_pool_method', 'avg'))
    elif nd.layer_type == 'attn':
        X = compute_attn_joint_arbors(nd.weight, li, nd.attn_weights,
                                      stimulus_weights=nd.stimulus_weights,
                                      stimulus_threshold=nd.stimulus_threshold)
    else:
        X = compute_joint_arbors_normalized(nd.weight, li,
                                            stimulus_weights=nd.stimulus_weights,
                                            stimulus_threshold=nd.stimulus_threshold)
    return np.clip(X, 0, None)


def act_matrix(nd):
    """Activation-only matrix (N, features) for the A1 baseline."""
    li = layer_inputs[nd.layer_idx]
    if nd.layer_type == 'attn' and getattr(nd, 'attn_weights', None) is not None:
        # same effective input the arbor sees: attention-weighted token mixture
        return np.einsum('nt,ntd->nd', nd.attn_weights, li)
    if li.ndim == 4:
        return li.mean(axis=(2, 3))                 # conv: pool spatial
    return li.reshape(len(li), -1)

  built: n_samples=9820  n_classes=2
  caps={'recon': False, 'roundtrip': False, 'ablation': False, 'attribution': True, 'bft_pixel': False, 'multi_seed': False}


## S1 · NMF factor stability + k-sensitivity

Are the factors reproducible across NMF random seeds (Hungarian-matched cosine), and how sensitive are they to the chosen rank K?

In [4]:
stab = {}
for li in layer_ids:
    nd = nodes_by_layer[li]
    off = stab_mean(node_arbor_pos(nd), nd.img_factors.shape[1])
    stab[li] = {'mean': float(off.mean()), 'std': float(off.std()),
                'k': int(nd.img_factors.shape[1])}
nd0 = nodes_by_layer[layer_ids[0]]
ksens, _ = compute_k_sensitivity(sub_rows(node_arbor_pos(nd0)), nd0.img_factors.shape[1],
                                 n_seeds=3, max_iter=AUX_MAX_ITER)
results['stability'] = {'per_layer': stab,
                        'k_sensitivity': {k: jsonable(v) for k, v in ksens.items()}}

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar([f'L{li}' for li in layer_ids], [stab[li]['mean'] for li in layer_ids],
       yerr=[stab[li]['std'] for li in layer_ids], capsize=3, color='#4e79a7')
ax.axhline(0.9, ls='--', c='#e15759', lw=1)
ax.set(ylim=(0, 1.05), ylabel='pairwise cosine sim', title=f'NMF stability — {EXP}')
fig.tight_layout(); savefig(fig, 'fig_nmf_stability.pdf')
print('  per-layer stability:', {li: round(stab[li]['mean'], 3) for li in layer_ids})

  saved figs/09_validation/vit_mnist/fig_nmf_stability.pdf
  per-layer stability: {0: 0.854, 1: 0.908, 2: 0.887, 3: 0.999}


## S2 · Causal reconstruction fidelity *(primary-mode only)*

Replace a layer's real pre-activation with its factor reconstruction and re-run the model. Headline metrics are **pre-activation R²** and **absolute CE gap** — the raw CE ratio explodes when the real loss is ~0, so it is reported but not relied on.

In [5]:
if caps['recon']:
    vs = tree.validation_summary()
    results['recon'] = {'overall': jsonable(vs['overall']), 'per_layer': jsonable(vs['per_layer'])}
    lids = sorted(vs['per_layer'])
    fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
    for a, m, t in [(ax[0], 'preact_r2', 'pre-activation R²'), (ax[1], 'abs_ce_gap', 'abs CE gap')]:
        a.bar([f'L{li}' for li in lids], [vs['per_layer'][li][m]['mean'] for li in lids],
              color='#4e79a7')
        a.set_title(t, fontsize=10)
    fig.suptitle(f'Causal reconstruction — {EXP}', y=1.03)
    fig.tight_layout(); savefig(fig, 'fig_recon_validation.pdf')
    print('  preact_R2 median=%.3f  min=%.3f' % (vs['overall']['preact_r2']['median'],
                                                 vs['overall']['preact_r2']['min']))
else:
    results['recon'] = 'skipped: needs primary-mode BFT (layer-dict archs have no model hook)'
    print('skipped — layer-dict mode')

skipped — layer-dict mode


## S3 · NNLS projection round-trip

Project the traced stimuli back onto the fixed factors and measure fingerprint recovery.

In [6]:
if caps['roundtrip']:
    try:
        proj = project_stimuli_onto_tree(tree, layer_inputs)
        Fo = extract_fingerprint_matrix(tree, np.arange(n_samples))
        Fr = extract_fingerprint_matrix(proj, np.arange(n_samples))
        rt = 1.0 - paired_cosine_distances(Fo, Fr)
        results['roundtrip'] = {'mean': float(rt.mean()), 'std': float(rt.std()),
                                'min': float(rt.min())}
        fig, ax = plt.subplots(figsize=(4.6, 3.1))
        ax.hist(rt, bins=30, color='#59a14f')
        ax.set(title=f'NNLS round-trip — {EXP}', xlabel='cosine similarity')
        fig.tight_layout(); savefig(fig, 'fig_nnls_roundtrip.pdf')
        print('  round-trip cosine %.3f' % rt.mean())
    except Exception as e:
        results['roundtrip'] = f'error: {e}'
        print('  round-trip failed:', e)
else:
    results['roundtrip'] = 'skipped: NNLS projection not supported through attn nodes'
    print('skipped — attn nodes')

skipped — attn nodes


## S4 · Fingerprint separability (task + fine-grained)

Silhouette and kNN accuracy of BFT fingerprints vs raw activations. The **fine-grained** label (e.g. the 4 digits behind even/odd) is the more informative test when the task label is trivially separable.

In [7]:
F = extract_fingerprint_matrix(tree, np.arange(n_samples))
# Activation baseline: use act_matrix so conv feature maps are spatially POOLED and
# attn tokens are collapsed by the CLS weights. Flattening instead would be both
# unfair and enormous (SqueezeNet spine maps flatten to ~1e6 dims/sample).
_alayers = layer_ids[1:] if len(layer_ids) > 1 else layer_ids
A = np.concatenate([act_matrix(nodes_by_layer[i]) for i in _alayers], axis=1)

sep = {'by_task': {}, 'by_fine': {}}
for name, X in [('bft_fingerprint', F), ('raw_activations', A)]:
    s, k = sep_metrics(X, targets);  sep['by_task'][name] = {'silhouette': s, 'knn_acc': k}
    s2, k2 = sep_metrics(X, fine);   sep['by_fine'][name] = {'silhouette': s2, 'knn_acc': k2}
results['separability'] = sep

fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
emb = MDS(2, dissimilarity='precomputed', random_state=0,
          normalized_stress='auto').fit_transform(cosine_distances(F))
for c in np.unique(targets):
    ax[0].scatter(*emb[targets == c].T, s=6, alpha=0.5, label=str(ctx['class_names'].get(c, c)))
ax[0].set(title='MDS · BFT fingerprints'); ax[0].set_xticks([]); ax[0].set_yticks([])
if ctx['n_classes'] <= 4:
    ax[0].legend(fontsize=7)
xb = np.arange(2)
ax[1].bar(xb - 0.2, [sep['by_task']['bft_fingerprint']['knn_acc'],
                     sep['by_fine']['bft_fingerprint']['knn_acc']], 0.4,
          label='BFT fp', color='#4e79a7')
ax[1].bar(xb + 0.2, [sep['by_task']['raw_activations']['knn_acc'],
                     sep['by_fine']['raw_activations']['knn_acc']], 0.4,
          label='activations', color='#e15759')
ax[1].set_xticks(xb); ax[1].set_xticklabels(['task kNN', 'fine kNN'])
ax[1].set(ylim=(0, 1.05), title='class separability'); ax[1].legend(fontsize=8)
fig.tight_layout(); savefig(fig, 'fig_fingerprint_separability.pdf')
print('  task kNN  BFT %.3f vs act %.3f' % (sep['by_task']['bft_fingerprint']['knn_acc'],
                                            sep['by_task']['raw_activations']['knn_acc']))
print('  fine kNN  BFT %.3f vs act %.3f' % (sep['by_fine']['bft_fingerprint']['knn_acc'],
                                            sep['by_fine']['raw_activations']['knn_acc']))

  saved figs/09_validation/vit_mnist/fig_fingerprint_separability.pdf
  task kNN  BFT 0.979 vs act 0.988
  fine kNN  BFT 0.979 vs act 0.988


## S5 · A1 — weight×activation arbor vs activation-only NMF

The key novelty control: does multiplying in the weights buy anything over factorizing activations alone (≈ CRAFT/ICE)? Compares stability, class selectivity, and fingerprint separability.

In [8]:
def class_selectivity(W, y):
    """Bounded: max over factors and classes of one-vs-rest ROC-AUC (0.5 = none)."""
    best = 0.5
    for k in range(W.shape[1]):
        if W[:, k].std() < 1e-12:
            continue
        for c in np.unique(y):
            a = roc_auc_score((y == c).astype(int), W[:, k])
            best = max(best, a, 1 - a)
    return float(best)


a1, afp, cfp = {'per_layer': {}}, [], []
for li in layer_ids:
    nd = nodes_by_layer[li]
    k  = nd.img_factors.shape[1]
    Xa = node_arbor_pos(nd)
    Ac = np.clip(act_matrix(nd), 0, None)
    kA = max(1, min(k, Ac.shape[1]))
    Wa, _ = fit_nmf(Xa, k)
    Wc, _ = fit_nmf(Ac, kA)
    a1['per_layer'][li] = {'stability_arbor': float(stab_mean(Xa, k).mean()),
                           'stability_act':   float(stab_mean(Ac, kA).mean()),
                           'selectivity_arbor': class_selectivity(Wa, targets),
                           'selectivity_act':   class_selectivity(Wc, targets)}
    afp.append(Wa); cfp.append(Wc)

sA, kA_ = sep_metrics(np.concatenate(afp, 1), targets)
sC, kC_ = sep_metrics(np.concatenate(cfp, 1), targets)
a1['fingerprint_separability'] = {'arbor_nmf': {'silhouette': sA, 'knn_acc': kA_},
                                  'activation_nmf': {'silhouette': sC, 'knn_acc': kC_}}
results['A1_weight_vs_activation'] = a1

fig, ax = plt.subplots(1, 3, figsize=(12, 3.3))
xl = np.arange(len(layer_ids))
for a, key, t in [(ax[0], 'stability', 'NMF stability'),
                  (ax[1], 'selectivity', 'class selectivity (AUC)')]:
    a.bar(xl - 0.2, [a1['per_layer'][li][f'{key}_arbor'] for li in layer_ids], 0.4,
          label='W·a arbor (BFT)', color='#4e79a7')
    a.bar(xl + 0.2, [a1['per_layer'][li][f'{key}_act'] for li in layer_ids], 0.4,
          label='activation-only', color='#e15759')
    a.set_xticks(xl); a.set_xticklabels([f'L{li}' for li in layer_ids])
    a.set_title(t, fontsize=10); a.legend(fontsize=8)
ax[2].bar([0, 1], [kA_, kC_], color=['#4e79a7', '#e15759'])
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(['arbor', 'act-only'])
ax[2].set(ylim=(0, 1.05), title='fingerprint kNN')
fig.suptitle(f'A1 — weight×activation vs activation-only ({EXP})', y=1.04)
fig.tight_layout(); savefig(fig, 'fig_a1_weight_vs_activation.pdf')
print('  selectivity (arbor vs act):',
      {li: (round(a1['per_layer'][li]['selectivity_arbor'], 3),
            round(a1['per_layer'][li]['selectivity_act'], 3)) for li in layer_ids})

  saved figs/09_validation/vit_mnist/fig_a1_weight_vs_activation.pdf
  selectivity (arbor vs act): {0: (0.913, 0.814), 1: (0.915, 0.857), 2: (0.85, 0.828), 3: (0.895, 0.953)}


## S6 · Causal circuit ablation

MLPs use the validated **weight-level** `ablation_sweep`. Non-MLP architectures use a coarser **filter-level proxy** (zeroing whole output units of the module each BFT node names) — labelled as such in the figure and JSON.

In [9]:
if caps['ablation']:
    el   = ctx['eval_loader']
    cap  = 1500 if MODE == 'local' else len(el.dataset)
    eloader = DataLoader(Subset(el.dataset, list(range(min(cap, len(el.dataset))))), batch_size=256)
    FRAC = 0.20
    classes = list(range(ctx['n_classes']))
    if MODE == 'local':
        classes = classes[:min(4, len(classes))]
    tdrop, bdrop, rdrop = [], [], []

    if hasattr(model, 'linear_layer_indices'):
        # MLP: validated library weight-level ablation (class-specific).
        from src import ablation_sweep
        method_kind = 'library weight-level (bft_top)'
        for d in classes:
            ab = ablation_sweep(model, tree, eloader, target_class=d, fractions=[FRAC],
                                methods=['bft_top', 'random'],
                                label_transform=ctx['label_transform'], device=DEVICE,
                                n_random_repeats=3, verbose=0)
            others = [c for c in classes if c != d]
            tdrop.append(ab.baseline[d] - ab.results['bft_top'][FRAC][d])
            bdrop.append(np.mean([ab.baseline[c] - ab.results['bft_top'][FRAC][c] for c in others]))
            rdrop.append(ab.baseline[d] - ab.results['random'][FRAC][d])
    else:
        # CNN / SqueezeNet: generic filter-level proxy — zeros whole output units of the
        # module named by each BFT node. Coarser than weight-level; label it as such.
        method_kind = 'generic filter-level proxy'
        name_by_layer = {nd.layer_idx: nd.layer_name for nd in tree.nodes()}
        base = per_class_accuracy(model, eloader, ctx['label_transform'], DEVICE)

        def ablate_units(ubl):
            m2 = copy.deepcopy(model); mm = dict(m2.named_modules())
            with torch.no_grad():
                for l, us in ubl.items():
                    for i in us:
                        mm[name_by_layer[l]].weight[i] = 0
            return m2

        for d in classes:
            sc, _ = select_class_circuit(tree, targets, d)
            agg = {}
            for (l, i, j), v in sc.items():
                agg.setdefault(l, {}); agg[l][i] = agg[l].get(i, 0.0) + abs(v)
            allu = sorted([(l, i, m) for l, dd in agg.items() for i, m in dd.items()],
                          key=lambda x: -x[2])
            nab, top = max(1, int(FRAC * len(allu))), {}
            for l, i, _ in allu[:nab]:
                top.setdefault(l, []).append(i)
            rnd = {l: list(RNG.choice(list(agg[l]), min(len(v), len(agg[l])), replace=False))
                   for l, v in top.items()}
            at = per_class_accuracy(ablate_units(top), eloader, ctx['label_transform'], DEVICE)
            ar = per_class_accuracy(ablate_units(rnd), eloader, ctx['label_transform'], DEVICE)
            others = [c for c in classes if c != d]
            tdrop.append(base[d] - at[d])
            bdrop.append(np.mean([base[c] - at[c] for c in others]))
            rdrop.append(base[d] - ar[d])

    w_p  = (wilcoxon(tdrop, bdrop).pvalue if len(tdrop) >= 2
            and np.any(np.array(tdrop) != np.array(bdrop)) else float('nan'))
    tt_p = ttest_rel(tdrop, rdrop).pvalue if len(tdrop) >= 2 else float('nan')
    results['ablation'] = {'method': method_kind, 'frac': FRAC, 'n_classes_eval': len(classes),
                           'bft_target_drop_mean': float(np.mean(tdrop)),
                           'bft_bystander_drop_mean': float(np.mean(bdrop)),
                           'random_target_drop_mean': float(np.mean(rdrop)),
                           'wilcoxon_target_vs_bystander_p': float(w_p),
                           'ttest_bft_vs_random_p': float(tt_p)}
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    ax.bar([0, 1, 2], [np.mean(tdrop), np.mean(bdrop), np.mean(rdrop)],
           yerr=[np.std(tdrop), np.std(bdrop), np.std(rdrop)], capsize=3,
           color=['#e15759', '#4e79a7', '#9c755f'])
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['BFT target', 'BFT bystander', 'random target'])
    ax.set(ylabel=f'accuracy drop @ {int(FRAC * 100)}%',
           title=f'Circuit ablation — {EXP}\n({method_kind}; Wilcoxon p={w_p:.2g})')
    fig.tight_layout(); savefig(fig, 'fig_ablation_pruning.pdf')
    print('  target %.3f | bystander %.3f | random %.3f  (Wilcoxon p=%.3g)' % (
        np.mean(tdrop), np.mean(bdrop), np.mean(rdrop), w_p))
else:
    results['ablation'] = 'skipped: BFT node names do not map onto model weight modules'
    print('skipped — no name-mappable weight modules')

skipped — no name-mappable weight modules


## S7 · Pixel attribution baselines (Captum)

Integrated Gradients / Saliency / input-magnitude, plus the BFT input-layer map where the input layer is pixel-shaped (MLPs). Scored by **class discriminability**: kNN accuracy predicting the class from a per-sample attribution map.

In [10]:
if caps['attribution']:
    from captum.attr import IntegratedGradients, Saliency

    class _Wrap(nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m

        def forward(self, x):
            o = self.m(x)
            return o[0] if isinstance(o, tuple) else o

    wrap = _Wrap(model).to(DEVICE).eval()
    imgs = torch.as_tensor(ctx['images'], dtype=torch.float32, device=DEVICE)
    tg   = torch.as_tensor(targets, dtype=torch.long, device=DEVICE)
    D    = int(np.prod(ctx['images'].shape[1:]))

    ig, sal = IntegratedGradients(wrap), Saliency(wrap)
    ig_m, sl_m, B = np.zeros((n_samples, D)), np.zeros((n_samples, D)), 128
    for s in range(0, n_samples, B):
        xb = imgs[s:s + B].clone().requires_grad_(True); yb = tg[s:s + B]
        ig_m[s:s + B] = ig.attribute(xb, target=yb, n_steps=IG_STEPS
                                     ).abs().reshape(len(yb), -1).detach().cpu().numpy()
        xb2 = imgs[s:s + B].clone().requires_grad_(True)
        sl_m[s:s + B] = sal.attribute(xb2, target=yb).abs().reshape(len(yb), -1).detach().cpu().numpy()

    attr = {'IG': ig_m, 'Saliency': sl_m,
            'input_mag': np.abs(ctx['images'].reshape(n_samples, -1))}
    if caps['bft_pixel']:
        bm = np.zeros((n_samples, D))
        for nd in tree.nodes():
            if nd.layer_idx == layer_ids[0]:
                rec = nd.img_factors @ nd.connection_factors.T
                bm += np.abs(rec.reshape(n_samples, nd.weight.shape[0], -1).sum(1))
        attr['BFT'] = bm

    def disc(M, y):
        Xn = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
        return knn_cv(Xn, y)

    dt = {k: disc(v, targets) for k, v in attr.items()}
    df = {k: disc(v, fine) for k, v in attr.items()}
    results['attribution'] = {'discriminability_task': dt, 'discriminability_fine': df,
                              'metric': 'kNN 3-fold CV accuracy from per-sample attribution map'}
    fig, ax = plt.subplots(figsize=(5.8, 3.2)); xm = np.arange(len(attr))
    ax.bar(xm - 0.2, [dt[k] for k in attr], 0.4, label='task', color='#4e79a7')
    ax.bar(xm + 0.2, [df[k] for k in attr], 0.4, label='fine', color='#e15759')
    ax.set_xticks(xm); ax.set_xticklabels(list(attr))
    ax.set(ylim=(0, 1), title=f'attribution discriminability — {EXP}'); ax.legend(fontsize=8)
    fig.tight_layout(); savefig(fig, 'fig_attribution_baselines.pdf')
    print('  discriminability (task):', {k: round(v, 3) for k, v in dt.items()})
else:
    results['attribution'] = 'skipped'
    print('skipped')

  saved figs/09_validation/vit_mnist/fig_attribution_baselines.pdf
  discriminability (task): {'IG': 0.995, 'Saliency': 0.964, 'input_mag': 0.98}


## S8 · Cross-seed circuit robustness *(needs ≥2 seed checkpoints)*

In [11]:
if caps['multi_seed']:
    Hs = []
    for s, m_s in ctx['seeds'].items():
        r_s = bft(m_s, ctx['vloader'], **ctx['bft_kwargs'], weighting='img_selectivity',
                  max_iter=AUX_MAX_ITER, n_jobs=3)
        H = r_s.root.img_factors
        Hs.append(H / (np.linalg.norm(H, axis=0, keepdims=True) + 1e-12))
    _, sc = align_factors(Hs[0], Hs[1:])
    results['seed_robustness'] = {'mean': float(np.mean(sc)),
                                  'scores': [float(x) for x in sc]}
    fig, ax = plt.subplots(figsize=(4.4, 3.1))
    ax.bar(range(len(sc)), sc, color='#59a14f')
    ax.axhline(np.mean(sc), ls='--', c='k')
    ax.set(ylim=(0, 1.05), xlabel='seed (vs seed 0)', ylabel='root factor cosine sim',
           title=f'cross-seed robustness — {EXP}')
    fig.tight_layout(); savefig(fig, 'fig_seed_robustness.pdf')
    print('  root factor alignment %.3f' % np.mean(sc))
else:
    results['seed_robustness'] = 'skipped: <2 seed checkpoints on disk'
    print('skipped — need >=2 seed checkpoints')

skipped — need >=2 seed checkpoints


## FU1 · Rank / recon-threshold sweep

Per-layer sweep of NMF rank K vs stability, arbor reconstruction R², and class selectivity, recommending K* per layer. Converges the one hyperparameter the validation flagged: effective stable+reconstructive rank is below the k_max caps.

In [12]:
# FU1 — per-layer rank / recon-threshold sweep. For each traced layer, sweep the NMF rank K
# and record (i) init-stability, (ii) arbor reconstruction R², (iii) class selectivity, then
# recommend K* = smallest K with recon-R² >= R2_TARGET and stability >= STAB_TARGET (else the
# K that maximizes recon-R²). Local per-layer sweep — no full re-trace.
def arbor_r2(X, W, H):
    Xhat = W @ H.T
    ss_res = float(((X - Xhat) ** 2).sum()); ss_tot = float(((X - X.mean()) ** 2).sum())
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')


R2_TARGET, STAB_TARGET = 0.8, 0.85
fu1 = {}
for li in layer_ids:
    nd = nodes_by_layer[li]
    X = node_arbor_pos(nd)
    kcap = int(min(8, X.shape[1], max(6, nd.img_factors.shape[1] + 2)))
    rows = []
    for K in range(1, kcap + 1):
        W, H = fit_nmf(X, K)
        rows.append({'K': K, 'stability': float(stab_mean(X, K).mean()),
                     'recon_r2': arbor_r2(X, W, H), 'selectivity': class_selectivity(W, targets)})
    # arbor-R² is monotone increasing in K, stability decreasing — so K* is capped by
    # stability: stay within the stable ranks, then take the smallest that also reconstructs
    # well enough (else the most expressive still-stable rank). Falls back to K=1.
    stable = [r for r in rows if r['stability'] >= STAB_TARGET]
    if stable:
        good = [r['K'] for r in stable if r['recon_r2'] >= R2_TARGET]
        kstar = good[0] if good else max(r['K'] for r in stable)
    else:
        kstar = 1
    fu1[li] = {'default_k': int(nd.img_factors.shape[1]), 'k_star': int(kstar), 'sweep': rows}
    print(f'  L{li}: default k={nd.img_factors.shape[1]} -> K*={kstar}')
results['FU1_rank_sweep'] = {'r2_target': R2_TARGET, 'stab_target': STAB_TARGET, 'per_layer': fu1}

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for li in layer_ids:
    sw = fu1[li]['sweep']; Ks = [r['K'] for r in sw]
    ax[0].plot(Ks, [r['stability'] for r in sw], 'o-', label=f'L{li}')
    ax[1].plot(Ks, [r['recon_r2'] for r in sw], 'o-')
    ax[2].plot(Ks, [r['selectivity'] for r in sw], 'o-')
    ax[0].axvline(fu1[li]['k_star'], ls=':', alpha=0.3)
ax[0].axhline(STAB_TARGET, ls='--', c='gray'); ax[0].set(title='NMF stability', xlabel='K', ylim=(0, 1.05))
ax[1].axhline(R2_TARGET, ls='--', c='gray'); ax[1].set(title='arbor reconstruction R²', xlabel='K')
ax[2].set(title='class selectivity (AUC)', xlabel='K', ylim=(0.45, 1.02))
ax[0].legend(fontsize=7, ncol=2)
fig.suptitle(f'FU1 rank sweep — {EXP}  (dotted = recommended K*)', y=1.03)
fig.tight_layout(); savefig(fig, 'fig_fu1_rank_sweep.pdf')
print('  recommended K* per layer:', {li: fu1[li]['k_star'] for li in layer_ids})

  L0: default k=8 -> K*=4


  L1: default k=6 -> K*=8


  L2: default k=6 -> K*=5


  L3: default k=4 -> K*=6


  saved figs/09_validation/vit_mnist/fig_fu1_rank_sweep.pdf
  recommended K* per layer: {0: 4, 1: 8, 2: 5, 3: 6}


## FU2 · Causal reconstruction for CNN FC head / ViT FFN

Extends the causal-reconstruction axis (S2) to fc-type nodes of layer-dict architectures. Conv nodes stay out of scope (spatial pooling is not invertible); attn nodes excluded. On MLPs this cross-checks S2. **Untested on cluster for CNN/ViT.**

In [13]:
# FU2 — causal reconstruction for fc-type nodes in layer-dict architectures (CNN FC head,
# ViT FFN). Reuses src.recon_validation. Conv nodes are NOT reconstructable (spatial pooling
# is not invertible) and attn nodes are excluded. For MLPs this cross-checks S2's fc recon.
# UNTESTED ON CLUSTER for CNN/ViT — validated only via the MLP equivalence.
from src.recon_validation import reconstruct_preactivation, _forward_ce, CE_FLOOR
import torch as _torch

_OVERRIDE = {'B0-FFN1': 'block.ffn1', 'B0-FFN2': 'block.ffn2'}   # ViT tree-name -> module name
_mods = dict(model.named_modules())


def _resolve_fc(nd):
    name = _OVERRIDE.get(nd.layer_name, nd.layer_name)
    m = _mods.get(name)
    ok = m is not None and getattr(m, 'weight', None) is not None and m.weight.ndim == 2
    return (name, m) if ok else (None, None)


def recon_fc_node(nd):
    name, module = _resolve_fc(nd)
    if module is None:
        return None
    act_input = layer_inputs[nd.layer_idx]
    z, active = reconstruct_preactivation(nd.weight, act_input, nd.img_factors, nd.connection_factors,
                                          nd.neg_img_factors, nd.neg_connection_factors,
                                          nd.stimulus_weights, None, nd.stimulus_threshold)
    idx = np.where(active)[0]
    if len(idx) == 0:
        return None
    imgs, tg, zc = ctx['images'][idx], ctx['targets'][idx].astype(int), z[idx]
    bias = module.bias.detach() if getattr(module, 'bias', None) is not None else None
    off = {'i': 0}

    def hook(mod, inp, out):
        b = out.shape[0]
        zt = _torch.from_numpy(zc[off['i']:off['i'] + b]).to(out.device, out.dtype)
        if bias is not None:
            zt = zt + bias.to(out.device, out.dtype)
        off['i'] += b
        if out.dim() == 3:                      # (N,T,d): inject CLS-token reconstruction only
            out = out.clone(); out[:, 0, :] = zt; return out
        return zt                               # (N,d): full replace

    real = _forward_ce(model, imgs, tg, DEVICE)
    h = module.register_forward_hook(hook)
    try:
        recon = _forward_ce(model, imgs, tg, DEVICE)
    finally:
        h.remove()
    zt_true = act_input[idx] @ nd.weight.T
    ss_res = float(((zt_true - zc) ** 2).sum()); ss_tot = float(((zt_true - zt_true.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    return {'module': name, 'real_ce': float(real), 'recon_ce': float(recon),
            'abs_ce_gap': float(recon - real), 'loss_ratio_floored': float(recon / max(real, CE_FLOOR)),
            'preact_r2': r2, 'n_eval': int(len(idx))}


fu2 = {}
for nd in tree.nodes():
    if nd.layer_type != 'fc':
        continue
    r = recon_fc_node(nd)
    if r is not None:
        fu2[f'L{nd.layer_idx}:{r["module"]}'] = r
if fu2:
    results['FU2_recon_fc'] = fu2
    keys = list(fu2)
    fig, ax = plt.subplots(figsize=(5.8, 3.2))
    ax.bar(range(len(keys)), [fu2[k]['preact_r2'] for k in keys], color='#4e79a7')
    ax.axhline(1.0, ls='--', c='gray'); ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=30, ha='right', fontsize=7)
    ax.set(ylim=(None, 1.1), ylabel='pre-activation R²', title=f'FU2 fc-node reconstruction — {EXP}')
    fig.tight_layout(); savefig(fig, 'fig_fu2_recon_fc.pdf')
    print('  fc-node recon preact_r2:', {k: round(v['preact_r2'], 3) for k, v in fu2.items()})
else:
    results['FU2_recon_fc'] = 'skipped: no fc node maps onto a Linear module'
    print('  FU2 skipped — no resolvable fc node (e.g. all-conv ImageNet spine)')

  saved figs/09_validation/vit_mnist/fig_fu2_recon_fc.pdf
  fc-node recon preact_r2: {'L3:block.ffn2': 0.432, 'L2:block.ffn1': 0.456}


## S9 · Dump all results to JSON

In [14]:
results['figures'] = figpaths
results['caps']    = caps
results['config']  = {'n_trace': N_TRACE, 'bft_max_iter': BFT_MAX_ITER,
                      'aux_max_iter': AUX_MAX_ITER, 'stab_seeds': STAB_SEEDS,
                      'n_samples': n_samples, 'device': str(DEVICE)}
out = os.path.join(RES_DIR, f'nb09_{EXP}.json')
with open(out, 'w') as f:
    json.dump(jsonable(results), f, indent=2)
with open(out) as f:
    json.load(f)                      # round-trip check
print('wrote', os.path.relpath(out, REPO))
print(json.dumps(jsonable({k: v for k, v in results.items()
                           if k not in ('figures', 'config')}), indent=2)[:2500])

wrote data/results/nb09_vit_mnist.json
{
  "experiment": "vit_mnist",
  "mode": "cluster",
  "stability": {
    "per_layer": {
      "0": {
        "mean": 0.8539475626415677,
        "std": 0.03204108234293844,
        "k": 8
      },
      "1": {
        "mean": 0.908323143588172,
        "std": 0.07417662957659937,
        "k": 6
      },
      "2": {
        "mean": 0.8865364578035143,
        "std": 0.05372252100985539,
        "k": 6
      },
      "3": {
        "mean": 0.9991787552833558,
        "std": 0.0009030969136062813,
        "k": 4
      }
    },
    "k_sensitivity": {
      "k_star": [
        1.0,
        1.0,
        1.0,
        1.0,
        1.0,
        1.0,
        1.0,
        1.0
      ],
      "k_minus1": [
        0.7683608531951904,
        0.913425087928772,
        0.9073427319526672,
        0.985347330570221,
        0.986059308052063,
        0.9891452789306641,
        0.9071692824363708,
        0.9859874248504639
      ],
      "k_plus1": [
        0

## How to run on the cluster (GPU)

The notebook is **parametric**: one execution = one experiment, selected by `NB09_EXP`.
`NB09_MODE=cluster` turns off all the laptop caps (uses all samples, 500 NMF iters,
10 stability seeds, 48 IG steps).

```bash
# one experiment, in-place execution with outputs saved into the notebook
cd notebooks
NB09_EXP=mlp_even_odd NB09_MODE=cluster \
  ../.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=100000 09_validation_all_models.ipynb
```

To keep one executed copy per experiment (recommended — otherwise each run overwrites
the previous outputs):

```bash
for EXP in mlp_even_odd mlp_digit cnn_cifar vit_mnist imagenet_cnn; do
  NB09_EXP=$EXP NB09_MODE=cluster \
    ../.venv/bin/jupyter nbconvert --to notebook --execute \
    --output "executed_09_${EXP}.ipynb" \
    --ExecutePreprocessor.timeout=100000 09_validation_all_models.ipynb
done
```

### SLURM example (one job per experiment)

```bash
#!/bin/bash
#SBATCH --job-name=bft-val
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --time=08:00:00
#SBATCH --array=0-4
EXPS=(mlp_even_odd mlp_digit cnn_cifar vit_mnist imagenet_cnn)
EXP=${EXPS[$SLURM_ARRAY_TASK_ID]}

cd "$SLURM_SUBMIT_DIR/notebooks"
source ../.venv/bin/activate
export NB09_EXP=$EXP NB09_MODE=cluster
jupyter nbconvert --to notebook --execute \
  --output "executed_09_${EXP}.ipynb" \
  --ExecutePreprocessor.timeout=100000 09_validation_all_models.ipynb
```

Outputs per experiment: `figs/09_validation/<EXP>/*.pdf` and
`data/results/nb09_<EXP>.json`. Send me the JSONs and I can iterate on the numbers.

### Prerequisites per experiment

| EXP | needs | status |
|---|---|---|
| `mlp_even_odd` | `data/models/mnist_even_odd_mlp_8_4_0134_seed{0..4}` | ✅ present |
| `mlp_digit` | `data/models/mnist_digit_mlp_40_20_seed0` (seeds 1–4 optional, enable S8) | ✅ seed 0 present |
| `cnn_cifar` | `data/models/cifar10_cnn_seed0` + CIFAR-10 | ✅ present |
| `vit_mnist` | nothing — trains TinyViT seed 0 inline (30 epochs) if no checkpoint | trains on first run |
| `imagenet_cnn` | **ImageNet val data** at `data/val/` (ImageFolder) or a torchvision ImageNet root at `data/`; SqueezeNet weights download automatically | ⚠️ data not present |

`pip install -r requirements.txt` (adds `captum`). GPU matters most for `cnn_cifar` and
`imagenet_cnn`; the MLPs and TinyViT run fine on CPU.

### Validation status — read this

| section | mlp_even_odd | mlp_digit | cnn_cifar | vit_mnist | imagenet_cnn |
|---|---|---|---|---|---|
| S1 stability | ✅ | ✅ | untested | untested | untested |
| S2 recon | ✅ | ✅ | skipped (layer-dict) | skipped | skipped |
| S3 round-trip | ✅ | ✅ | untested | skipped (attn) | untested |
| S4 separability | ✅ | ✅ | untested | untested | untested |
| S5 A1 | ✅ | ✅ | untested | untested | untested |
| S6 ablation | ✅ weight-level | ✅ weight-level | untested (proxy) | skipped | untested (proxy) |
| S7 attribution | ✅ | ✅ | untested | untested | untested |
| S8 seed robustness | ✅ | skipped (1 seed) | skipped | skipped | skipped |

**✅ = actually executed locally.** Everything marked *untested* is written from the
notebook 03/04/05 specs but has never been run — expect to fix small things on the first
cluster run (most likely: ViT `capture=True` attribute names, and the ImageNet data path).
Run `mlp_even_odd` first as a smoke test, then `cnn_cifar`, then the rest.